# 04 — Valutazione: calcolo del pass@1 su HumanEval

Notebook per eseguire il codice generato nel notebook 03 contro i test nascosti di
HumanEval, e calcolare pass@1 per ciascuno dei 13 modelli (baseline fp16 + 12 quantizzati).

Usiamo l'harness ufficiale `human-eval di OpenAI

In [1]:
import os
import json
import inspect
import pandas as pd

from human_eval.data import read_problems
from human_eval.evaluation import evaluate_functional_correctness

RESULTS_DIR = "../results"
GENERATIONS_DIR = f"{RESULTS_DIR}/generations"

CALIBRATIONS = ["random", "mixed", "code"]
QUANT_LEVELS = ["Q8_0", "Q4_K_M", "Q3_K_M", "Q2_K"]

MODEL_NAMES = ["baseline-f16"] + [f"{c}-{l}" for c in CALIBRATIONS for l in QUANT_LEVELS]

print(f"Modelli da valutare: {len(MODEL_NAMES)}")
for name in MODEL_NAMES:
    path = f"{GENERATIONS_DIR}/{name}.jsonl"
    exists = "OK" if os.path.exists(path) else "MANCANTE"
    print(f"  [{exists}] {name}")

Modelli da valutare: 13
  [OK] baseline-f16
  [OK] random-Q8_0
  [OK] random-Q4_K_M
  [OK] random-Q3_K_M
  [OK] random-Q2_K
  [OK] mixed-Q8_0
  [OK] mixed-Q4_K_M
  [OK] mixed-Q3_K_M
  [OK] mixed-Q2_K
  [OK] code-Q8_0
  [OK] code-Q4_K_M
  [OK] code-Q3_K_M
  [OK] code-Q2_K


## 2. Sblocco dell'esecuzione del codice

Per sicurezza, `human-eval` esegue codice generato da un modello (potenzialmente non affidabile).
Nelle versioni storiche della libreria la riga `exec(check_program, exec_globals)` in
`human_eval/execution.py` è commentata di default, e va sbloccata manualmente dopo aver letto
l'avviso di sicurezza.

La cella seguente verifica automaticamente lo stato del file: se la riga è già attiva non fa
nulla, altrimenti la scommenta. In ogni caso, stampa il contenuto della sezione rilevante così puoi
verificare a occhio cosa succede.

In [2]:
execution_file = inspect.getfile(__import__("human_eval.execution", fromlist=["execution"]))
print("File execution.py individuato in:", execution_file)

with open(execution_file, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Troviamo la riga che contiene la chiamata exec(check_program, ...)
target_idx = None
for i, line in enumerate(lines):
    if "exec(check_program" in line:
        target_idx = i
        break

if target_idx is None:
    raise RuntimeError("Riga 'exec(check_program...' non trovata: verificare manualmente il file.")

print("\n--- Contesto attorno alla riga trovata ---")
start = max(0, target_idx - 3)
end = min(len(lines), target_idx + 2)
for i in range(start, end):
    marker = ">>>" if i == target_idx else "   "
    print(f"{marker} {i+1}: {lines[i].rstrip()}")

is_commented = lines[target_idx].strip().startswith("#")
print(f"\nLa riga risulta {'COMMENTATA (esecuzione disattivata)' if is_commented else 'GIA ATTIVA'}.")

File execution.py individuato in: /Users/rubengigante/ProgettoTirocinio/.venv/lib/python3.12/site-packages/human_eval/execution.py

--- Contesto attorno alla riga trovata ---
    33:             exec_globals = {}
    34:             with swallow_io():
    35:                 with time_limit(timeout):
>>> 36:                     exec(check_program, exec_globals)
    37:             result.append("passed")

La riga risulta GIA ATTIVA.


In [3]:
if is_commented:
    # Rimuoviamo solo il carattere '#' iniziale (e uno spazio eventuale), preservando l'indentazione
    original_line = lines[target_idx]
    indent = original_line[:len(original_line) - len(original_line.lstrip())]
    stripped = original_line.strip().lstrip("#").lstrip()
    lines[target_idx] = f"{indent}{stripped}\n"

    with open(execution_file, "w", encoding="utf-8") as f:
        f.writelines(lines)

    print("Riga scommentata. Esecuzione del codice ora abilitata.")
    print("Nuova riga:", lines[target_idx].rstrip())
else:
    print("Nessuna modifica necessaria: l'esecuzione era già abilitata.")

Nessuna modifica necessaria: l'esecuzione era già abilitata.


## 3. Caricamento dei problemi HumanEval

`read_problems()` carica i 164 problemi con prompt, test nascosti ed entry point, dal file
bundled nella libreria stessa (indipendente dal dataset Hugging Face usato nel notebook 03,
ma corrispondono agli stessi problemi).

In [4]:
problems = read_problems()
print(f"Numero di problemi caricati: {len(problems)}")

# Controllo di coerenza: confrontiamo il numero di problemi con quello del notebook 03
assert len(problems) == 164, "Numero di problemi inatteso: verificare l'installazione di human-eval."

Numero di problemi caricati: 164


## 4. Valutazione dei 13 modelli

Per ciascun file `.jsonl` di generazioni, `evaluate_functional_correctness`:
- esegue ogni completamento contro i test del problema corrispondente,
- scrive un file di dettaglio `<nome_modello>.jsonl_results.jsonl` (passato/fallito per ogni problema),
- restituisce un dizionario con `pass@1`.

Con un solo completamento per problema (k=1, come generato nel notebook 03), valutiamo solo
`pass@1`.

In [5]:
evaluation_results = []

for name in MODEL_NAMES:
    sample_file = f"{GENERATIONS_DIR}/{name}.jsonl"

    if not os.path.exists(sample_file):
        print(f"=== {name}: file mancante, salto ===")
        evaluation_results.append({"model": name, "pass@1": None, "status": "ERRORE: file mancante"})
        continue

    print(f"\n=== Valutazione: {name} ===")
    try:
        result = evaluate_functional_correctness(
            sample_file=sample_file,
            k=[1],
            n_workers=4,
            timeout=10.0,
        )
        pass_at_1 = result["pass@1"]
        print(f"{name}: pass@1 = {pass_at_1:.4f}")
        evaluation_results.append({"model": name, "pass@1": pass_at_1, "status": "OK"})
    except Exception as e:
        print(f"ERRORE durante la valutazione di {name}: {e}")
        evaluation_results.append({"model": name, "pass@1": None, "status": f"ERRORE: {e}"})

print("\nValutazione completata per tutti i modelli.")


=== Valutazione: baseline-f16 ===
Reading samples...


164it [00:00, 14708.36it/s]


Running test suites...


100%|██████████| 164/164 [00:05<00:00, 31.94it/s]


Writing results to ../results/generations/baseline-f16.jsonl_results.jsonl...


...

Running test suites...


100%|██████████| 164/164 [00:04<00:00, 33.38it/s]


Writing results to ../results/generations/code-Q2_K.jsonl_results.jsonl...


100%|██████████| 164/164 [00:00<00:00, 73380.19it/s]

code-Q2_K: pass@1 = 0.2073

Valutazione completata per tutti i modelli.


### Riepilogo grezzo della valutazione

In [6]:
for r in evaluation_results:
    pass1_str = f"{r['pass@1']:.4f}" if r["pass@1"] is not None else "N/D"
    print(f"{r['model']:<18} pass@1={pass1_str:<8} [{r['status']}]")

baseline-f16       pass@1=0.3902   [OK]
random-Q8_0        pass@1=0.3780   [OK]
random-Q4_K_M      pass@1=0.3415   [OK]
random-Q3_K_M      pass@1=0.3659   [OK]
random-Q2_K        pass@1=0.2134   [OK]
mixed-Q8_0         pass@1=0.3780   [OK]
mixed-Q4_K_M       pass@1=0.3171   [OK]
mixed-Q3_K_M       pass@1=0.3537   [OK]
mixed-Q2_K         pass@1=0.2134   [OK]
code-Q8_0          pass@1=0.3780   [OK]
code-Q4_K_M        pass@1=0.3415   [OK]
code-Q3_K_M        pass@1=0.3537   [OK]
code-Q2_K          pass@1=0.2073   [OK]


## 5. Tabella riassuntiva (calibration × livello di bit)

Riorganizziamo i risultati in una tabella pivot: righe = calibration dataset, colonne = livello
di quantizzazione, valori = pass@1. La baseline fp16 viene mostrata separatamente come riferimento.

In [7]:
df = pd.DataFrame(evaluation_results)

baseline_row = df[df["model"] == "baseline-f16"]
baseline_pass1 = baseline_row["pass@1"].values[0] if len(baseline_row) > 0 else None

quantized_df = df[df["model"] != "baseline-f16"].copy()
quantized_df["calibration"] = quantized_df["model"].apply(lambda x: x.split("-")[0])
quantized_df["level"] = quantized_df["model"].apply(lambda x: "-".join(x.split("-")[1:]))

pivot = quantized_df.pivot(index="calibration", columns="level", values="pass@1")
pivot = pivot[QUANT_LEVELS]  # ordina le colonne come da definizione originale
pivot = pivot.reindex(CALIBRATIONS)  # ordina le righe

print(f"Baseline fp16: pass@1 = {baseline_pass1:.4f}\n" if baseline_pass1 is not None else "Baseline non disponibile\n")
print(pivot)

Baseline fp16: pass@1 = 0.3902

level            Q8_0    Q4_K_M    Q3_K_M      Q2_K
calibration                                        
random       0.378049  0.341463  0.365854  0.213415
mixed        0.378049  0.317073  0.353659  0.213415
code         0.378049  0.341463  0.353659  0.207317


## 6. Salvataggio dei risultati

In [8]:
output_csv = f"{RESULTS_DIR}/pass_at_1_summary.csv"

# Salviamo sia la tabella pivot leggibile sia i dati grezzi completi (inclusa la baseline)
pivot.to_csv(output_csv)
df.to_csv(f"{RESULTS_DIR}/pass_at_1_raw.csv", index=False)

print(f"Tabella pivot salvata in: {output_csv}")
print(f"Dati grezzi salvati in: {RESULTS_DIR}/pass_at_1_raw.csv")

Tabella pivot salvata in: ../results/pass_at_1_summary.csv
Dati grezzi salvati in: ../results/pass_at_1_raw.csv
